In [ ]:
from torchgeo.trainers import PixelwiseRegressionTask
import torch
import pytorch_lightning as pl
import numpy as np
import rasterio
import cv2
from torch.utils.data import Dataset, DataLoader
from typing import List
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint
import torch.nn as nn
import os

#TODO: Process the data folder into a clipped 32x32 set as files
#TODO: Look at other example data modules
class LSTNowcaster(pl.LightningModule):
    def __init__(self, in_channels=5, learning_rate=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = PixelwiseRegressionTask(
            model="unet",
            backbone="resnet50",
            weights=True,
            in_channels=in_channels,
            num_outputs=1,
            loss="mse",
            lr=learning_rate
        )
        self.criterion = nn.MSELoss()
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        inputs = batch['input']
        targets = batch['target']
        mask = batch['mask']

        outputs = self(inputs)
        loss = self.criterion(outputs[mask], targets[mask])

        self.log('train_loss', loss,
                 on_step=True,
                 on_epoch=True,
                 prog_bar=True,
                 sync_dist=True)  # Add this parameter
        return loss

    def validation_step(self, batch, batch_idx):
        inputs = batch['input']
        targets = batch['target']
        mask = batch['mask']

        outputs = self(inputs)
        loss = self.criterion(outputs[mask], targets[mask])

        self.log('val_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        inputs = batch['input']
        targets = batch['target']
        mask = batch['mask']
        profile = batch['profile']
        file_path = batch['file_path']

        outputs = self(inputs)
        mse = self.criterion(outputs[mask], targets[mask])

        # Save prediction if it's the first batch
        if batch_idx == 0:
            predicted = outputs.cpu().numpy().squeeze()
            mask_np = mask.cpu().numpy().squeeze()
            predicted[~mask_np] = np.nan

            profile = profile[0]  # Get first item's profile
            profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

            output_filename = f"predicted_LST_batch_{batch_idx}.tif"
            with rasterio.open(output_filename, "w", **profile) as dst:
                dst.write(predicted.astype(np.float32), 1)

            print(f"Saved predicted LST raster: {output_filename}")

        return {'test_loss': mse, 'batch_size': inputs.size(0)}

    def test_epoch_end(self, outputs):
        avg_mse = torch.stack([x['test_loss'] for x in outputs]).mean()
        rmse = torch.sqrt(avg_mse)

        self.log('test_mse', avg_mse)
        self.log('test_rmse', rmse)

        print(f"Test RMSE: {rmse:.4f}")

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

class RasterDataset(Dataset):
    def __init__(self, file_list, transform=None, nodata_fill_value=-9999.0):
        """
        Args:
            file_list (list): List of dictionaries containing file paths for each channel
            transform (callable, optional): Optional transform to be applied on a sample
            nodata_fill_value (float, optional): The nodata value expected in the rasters
        """
        self.file_list = file_list
        self.transform = transform
        self.nodata_fill_value = nodata_fill_value

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        sample_files = self.file_list[idx]

        # Process input channels
        channels = []
        for key in ['Albedo.tif', 'DEM.tif', 'Land_Cover.tif', 'NDVI.tif', 'NDWI.tif']:
            with rasterio.open(sample_files[key]) as src:
                channel = src.read(1).astype(np.float32)
            channel = np.where(np.isnan(channel), 0.0, channel)
            channels.append(channel)

        # Dynamic resizing
        ref_shape = channels[0].shape
        fixed_channels = []
        for ch in channels:
            if ch.shape != ref_shape:
                ch_resized = cv2.resize(ch, (ref_shape[1], ref_shape[0]),
                                        interpolation=cv2.INTER_LINEAR)
                fixed_channels.append(ch_resized)
            else:
                fixed_channels.append(ch)
        channels = fixed_channels

        # Find the center crop dimensions divisible by 32
        h, w = ref_shape
        new_h = (h // 32) * 32
        new_w = (w // 32) * 32

        # Calculate the starting positions for center crop
        start_h = (h - new_h) // 2
        start_w = (w - new_w) // 2

        # Crop all channels
        cropped_channels = []
        for ch in channels:
            cropped_ch = ch[start_h:start_h + new_h, start_w:start_w + new_w]
            cropped_channels.append(cropped_ch)

        x = np.stack(cropped_channels, axis=0)

        # Process target (LST)
        with rasterio.open(sample_files['LST.tif']) as src:
            y = src.read(1).astype(np.float32)

        valid_mask = ~np.isnan(y)
        valid_mask = valid_mask & (y != self.nodata_fill_value)
        y = np.where(valid_mask, y, 0.0)

        if y.shape != ref_shape:
            y = cv2.resize(y, (ref_shape[1], ref_shape[0]),
                           interpolation=cv2.INTER_LINEAR)
            valid_mask = cv2.resize(valid_mask.astype(np.uint8),
                                    (ref_shape[1], ref_shape[0]),
                                    interpolation=cv2.INTER_NEAREST)
            valid_mask = valid_mask.astype(bool)

        y = y[start_h:start_h + new_h, start_w:start_w + new_w]
        valid_mask = valid_mask[start_h:start_h + new_h, start_w:start_w + new_w]

        y = np.expand_dims(y, axis=0)
        valid_mask = np.expand_dims(valid_mask, axis=0)

        sample = {
            'input': torch.from_numpy(x),
            'target': torch.from_numpy(y),
            'mask': torch.from_numpy(valid_mask)
        }

        if self.transform:
            sample = self.transform(sample)

        if sample['input'].shape[1:] != sample['target'].shape[1:]:
            raise ValueError("Mismatch between input and target spatial dimensions.")

        return sample

class RasterDataModule(pl.LightningDataModule):
    def __init__(
            self,
            data_dir: str,
            batch_size: int = 1,
            num_workers: int = 2,
            train_ratio: float = 0.8,
            transform=None,
            nodata_fill_value: float = -9999.0
    ):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.train_ratio = train_ratio
        self.transform = transform
        self.nodata_fill_value = nodata_fill_value
        self.train_files = []
        self.val_files = []
        self.test_files = []

    def setup(self, stage=None):
        if not self.train_files:  # Only prepare data if not already prepared
            self.prepare_data()

    def prepare_data(self):
        file_list = []
        albedo_files = []

        x_dir = os.path.join(self.data_dir, 'X', 'less5CloudCover')
        for file_path in self.get_file_paths(x_dir)[:300]:
            if 'Albedo' in file_path:
                albedo_files.append(file_path)

        for albedo_path in albedo_files:
            scene_files = [f for f in os.listdir(os.path.dirname(albedo_path))
                           if os.path.isfile(os.path.join(os.path.dirname(albedo_path), f))]

            raster_dict = {}
            for raster_file in scene_files:
                raster_path = os.path.join(os.path.dirname(albedo_path), raster_file)
                raster_dict[raster_file] = raster_path

            lst_path = albedo_path.replace('/X/', '/y/').replace('Albedo.tif', 'LST.tif')
            if os.path.exists(lst_path):  # Only add if LST file exists
                raster_dict['LST.tif'] = lst_path
                file_list.append(raster_dict)

        # Split datasets
        import random
        random.shuffle(file_list)
        train_size = int(len(file_list) * self.train_ratio)
        val_size = int((len(file_list) - train_size) / 2)

        self.train_files = file_list[:train_size]
        self.val_files = file_list[train_size:train_size + val_size]
        self.test_files = file_list[train_size + val_size:]

        print(f"Dataset splits - Train: {len(self.train_files)}, Val: {len(self.val_files)}, Test: {len(self.test_files)}")

    def get_file_paths(self, folder_path: str) -> List[str]:
        file_paths = []
        for root, _, files in os.walk(folder_path):
            for file in files:
                full_path = os.path.abspath(os.path.join(root, file))
                file_paths.append(full_path)
        return file_paths

    def train_dataloader(self):
        return DataLoader(
            RasterDataset(self.train_files, self.transform, self.nodata_fill_value),
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )

    def val_dataloader(self):
        return DataLoader(
            RasterDataset(self.val_files, self.transform, self.nodata_fill_value),
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers
        )

    def test_dataloader(self):
        return DataLoader(
            RasterDataset(self.test_files, self.transform, self.nodata_fill_value),
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers
        )

def main():
    # Force CPU usage
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

    # Verify we're on CPU
    print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
    # import logging
    # def device_info_filter(record):
    #     return "PU available: " not in record.getMessage()
    #
    # logging.getLogger("lightning.pytorch.utilities.rank_zero").addFilter(device_info_filter)

    # Initialize data module
    data_module = RasterDataModule(
        data_dir="./Data",
        batch_size=1,
        num_workers=0
    )

    # Prepare data explicitly
    data_module.prepare_data()
    data_module.setup()



    # Initialize trainer with explicit steps
    trainer = pl.Trainer(
        max_epochs=200,
        gradient_clip_val=0.5,
        log_every_n_steps=1,
        enable_progress_bar=True,
        enable_model_summary=True,
        deterministic=True,
        num_sanity_val_steps=2,
        reload_dataloaders_every_n_epochs=1
    )

    model = LSTNowcaster(in_channels=5, learning_rate=1e-4)

    # Train model
    trainer.fit(model=model, datamodule=data_module)

    # Test model
    # trainer.test(model=model, datamodule=data_module)

if __name__ == "__main__":
    # Suppress specific warning
    # with warnings.catch_warnings():
    #     warnings.filterwarnings("ignore", message="TORCH_MODEL_ZOO is deprecated")
    main()

#TODO: Get an MNIST Training loop to work.

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch import nn, optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch.utils.data import random_split
import pytorch_lightning as pl
import torchmetrics
from torchmetrics import Metric


class MyAccuracy(Metric):
    def __init__(self):
        super().__init__()
        self.add_state("total", default=torch.tensor(0), dist_reduce_fx="sum")
        self.add_state("correct", default=torch.tensor(0), dist_reduce_fx="sum")

    def update(self, preds, target):
        preds = torch.argmax(preds, dim=1)
        assert preds.shape == target.shape
        self.correct += torch.sum(preds == target)
        self.total += target.numel()

    def compute(self):
        return self.correct.float() / self.total.float()


class NN(pl.LightningModule):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 50)
        self.fc2 = nn.Linear(50, num_classes)
        self.loss_fn = nn.CrossEntropyLoss()
        self.accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.my_accuracy = MyAccuracy()
        self.f1_score = torchmetrics.F1Score(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

    def training_step(self, batch, batch_idx):
        loss, scores, y = self._common_step(batch, batch_idx)
        accuracy = self.my_accuracy(scores, y)
        f1_score = self.f1_score(scores, y)
        self.log_dict({'train_loss': loss, 'train_accuracy': accuracy, 'train_f1_score': f1_score},
                      on_step=False, on_epoch=True, prog_bar=True)
        return {'loss': loss, "scores": scores, "y": y}

    def validation_step(self, batch, batch_idx):
        loss, scores, y = self._common_step(batch, batch_idx)
        self.log('val_loss', loss)
        return loss

    def test_step(self, batch, batch_idx):
        loss, scores, y = self._common_step(batch, batch_idx)
        self.log('test_loss', loss)
        return loss

    def _common_step(self, batch, batch_idx):
        x, y = batch
        x = x.reshape(x.size(0), -1)
        scores = self.forward(x)
        loss = self.loss_fn(scores, y)
        return loss, scores, y

    def predict_step(self, batch, batch_idx):
        x, y = batch
        x = x.reshape(x.size(0), -1)
        scores = self.forward(x)
        preds = torch.argmax(scores, dim=1)
        return preds

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=0.001)


class MnistDataModule(pl.LightningDataModule):
    def __init__(self, data_dir, batch_size, num_workers):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers

    def prepare_data(self):
        datasets.MNIST(self.data_dir, train=True, download=True)
        datasets.MNIST(self.data_dir, train=False, download=True)

    def setup(self, stage):
        entire_dataset = datasets.MNIST(
            root=self.data_dir,
            train=True,
            transform=transforms.ToTensor(),
            download=False,
        )
        self.train_ds, self.val_ds = random_split(entire_dataset, [50000, 10000])
        self.test_ds = datasets.MNIST(
            root=self.data_dir,
            train=False,
            transform=transforms.ToTensor(),
            download=False,
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=False,
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_ds,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=False,
        )

# Set device cuda for GPU if it's available otherwise run on the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
input_size = 784
num_classes = 10
learning_rate = 0.001
batch_size = 64
num_epochs = 3

model = NN(input_size=input_size, num_classes=num_classes)
dm = MnistDataModule(data_dir="dataset/", batch_size=batch_size, num_workers=4)
trainer = pl.Trainer(accelerator="gpu", devices=1, min_epochs=1, max_epochs=3, precision=16)
trainer.fit(model, dm)
trainer.validate(model, dm)
trainer.test(model, dm)

In [ ]:
def save_prediction_and_truth(model, test_loader, test_file_list, device):
    """
    Save both prediction and ground truth from test loader as georeferenced TIFFs
    """
    model.eval()

    with torch.no_grad():
        # Get one sample
        sample = next(iter(test_loader))
        inputs = sample['input'].to(device)
        targets = sample['target'].to(device)
        mask = sample['mask'].to(device)

        # Get corresponding LST file path
        lst_tif_path = test_file_list[0]['LST.tif']

        # Convert tensors to numpy arrays
        mask_np = mask.cpu().numpy().squeeze()
        targets_np = targets.cpu().numpy().squeeze()

        # Get model prediction
        outputs = model(inputs)
        predicted_np = outputs.cpu().numpy().squeeze()

        # Apply mask to both prediction and ground truth
        predicted_np[~mask_np] = np.nan
        targets_np[~mask_np] = np.nan

        # Get geospatial metadata from original LST file
        with rasterio.open(lst_tif_path) as src:
            profile = src.profile.copy()
            profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

            # Denormalize values back to Fahrenheit
            # Using the same range as in the Normalize class
            predicted_np = predicted_np * (250 - (-50)) + (-50)  # Updated range
            targets_np = targets_np * (250 - (-50)) + (-50)      # Updated range

            # Save prediction
            pred_filename = "predicted_LST.tif"
            with rasterio.open(pred_filename, "w", **profile) as dst:
                dst.write(predicted_np.astype(np.float32), 1)

            # Save ground truth
            truth_filename = "ground_truth_LST.tif"
            with rasterio.open(truth_filename, "w", **profile) as dst:
                dst.write(targets_np.astype(np.float32), 1)

            # Calculate and print some statistics for valid pixels
            valid_mask = ~np.isnan(predicted_np)
            if valid_mask.any():
                mae = np.mean(np.abs(predicted_np[valid_mask] - targets_np[valid_mask]))
                rmse = np.sqrt(np.mean((predicted_np[valid_mask] - targets_np[valid_mask])**2))
                print(f"Mean Absolute Error: {mae:.2f}°F")
                print(f"Root Mean Square Error: {rmse:.2f}°F")

        print(f"Saved files:")
        print(f"Predictions: {pred_filename}")
        print(f"Ground Truth: {truth_filename}")
        print(f"Original LST: {lst_tif_path}")

In [ ]:
    # Assume model and test_loader are already defined
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load the trained model
    checkpoint = torch.load('best_model.pth')
    model = UNet(n_channels=5, bilinear=False).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])

    save_prediction_and_truth(model, test_loader, test_file_list, device)